# OPERA DISP-S1 Time Series Data for Landslides
---

Welcome! This notebook demonstrates how to download and visualize surface displacement time series data from the **OPERA DISP-S1** product. We use a real-world example of a slow-moving landslide to show how these products can track cm-scale movements over time.

### What is OPERA DISP-S1?
The **OPERA Level-3 Surface Displacement from Sentinel-1 (DISP-S1)** product provides geocoded displacement measurements over North America. Generated using advanced Interferometric Synthetic Aperture Radar (InSAR) processing, it helps track both natural (e.g., landslides, volcanoes, tectonics) and anthropogenic (e.g., subsidence from groundwater pumping) surface motion.

### Notebook Workflow
1. **Environment Setup**: Install necessary Python packages and helper tools.
2. **Area Selection (AOI)**: Choose your target location and frame.
3. **Data Download**: Pull cropped displacement data directly from NASA Earthdata.
4. **Interactive Analysis**: Select a reference point on a map to adjust measurements.
5. **Quality Review**: Inspect coherence and other metrics to verify data reliability.
6. **Export**: Save results as GeoTIFFs for GIS or GIFs for animations.

---

### Data Architecture: The "Ministack"
To keep the data manageable and accurate, DISP-S1 uses a **ministack** structure:

- **Cumulative Displacement**: Each file in a ministack measures the motion between a *fixed reference date* and a *secondary date*.
- **Stack Size**: A ministack contains up to **15 acquisitions** (approx. 3-6 months of data).
- **The Chain**: The last date of one ministack becomes the reference date for the next. This notebook automatically "rebases" these stacks into a single, continuous time series for your analysis.

- **Layers Included**:
    - `Displacement`: Surface displacement from the reference date to the secondary date, spatially referenced to an arbitrary high coherence/persistent scatterer pixel within the scene.
    - `Short wavelength displacement`, spatially filtered to remove large scale (> 35 km) signals such as atmospheric noise.
    - `Recommended mask`: A derived “recommended” mask for users who wish to remove possible low-quality results, where 0 indicates a bad pixel, 1 is a good pixel. Pixels are set to 0 for three reasons: 1. the pixel is marked as water in /water_mask, 2. the pixel is 0 in /connected_component_labels, 3. the pixel has low /temporal_coherence and low /phase_similarity.
    - `Temporal coherence`: A measure of the average misfit between the optimized phase linking results and the original multi-looked interferograms.
    - `Phase similarity`: a quality metric describing the median cosine similarity between the pixel and its neighbors.
    - `Estimated phase quality` – an alternate quality metric computed using a normalized Gaussian filter on the displacement image after re-wrapping and converting back to a complex image with unit magnitudes.
    - `Connected component labels`: integer labels produced by the phase unwrapper after recomputing costs on the output Displacement raster. Used to find reliable, spatially contiguous areas within the displacement image. Different labels may indicate the there are phase jumps between labels.
    - `Persistent scatterer mask`: A binary image indicating where the phase from a persistent scatterer (PS) was used instead of a multi-looked phase linking result.
    - `SHP Counts`: Counts of statistically homogeneous pixels (SHP) used during adaptive multi-looking.
    - `Time series residuals`: The resulting misfit from solving the inversion problem after unwrapping a network of interferograms. The units are in radians to make it easer to find 2pi offsets that may be corrected.
    - `Water mask`: The binary water mask used during processing, where 0 indicates water and 1 indicates land.

---

*For technical details, see the [OPERA Product Specification](https://d2pn8kiwq2w21t.cloudfront.net/documents/OPERA_DISP_S1_Final_Product_Spec.pdf). 

Notebook Contributors: A. Handwerger, B. Raimbault, M. G. Bato, S. Sangha, M. Govorcin, S. Staniewicz.

## Setup your conda environment

The following environment setup only needs to be performed **once**. To run this notebook, you need a specific environment with InSAR analysis tools.

Open your terminal and run:
```bash
# 1. Create the environment
> conda create -n opera_disp-s1

# 2. Activate it (required every time you open a new terminal)
> conda activate opera_disp-s1

# 3. Install core dependencies
> conda install -c conda-forge python==3.12 jupyter ipyleaflet
```

---
> [!TIP]
> The code cell below will automatically check for and install the remaining library dependencies (MintPy and disp-xr) if they are missing.

## Import Libraries

In [ ]:
### Automated Environment Check
# This cell ensures all required libraries are installed and ready to go.

import os
import subprocess
import sys
import importlib.util

def run(cmd):
    """Run a shell command silently."""
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def install_dependencies():
    """Clone and install MintPy and disp-xr if not already present."""
    # Check for MintPy
    if not os.path.exists("MintPy"):
        print("Cloning MintPy...")
        run("git clone https://github.com/insarlab/MintPy.git")
        run("pip install -e ./MintPy")
    
    # Check for disp-xr
    if not os.path.exists("disp-xr"):
        print("Cloning disp-xr...")
        run("git clone https://github.com/opera-adt/disp-xr.git")
        run("pip install -e ./disp-xr")
    
    # Add to path regardless of installation to ensure they are available
    sys.path.insert(0, os.path.abspath("disp-xr/src"))
    sys.path.insert(0, os.path.abspath("MintPy/src"))
    
    # Check if a core package is missing before running bulk pip install
    if importlib.util.find_spec("opera_utils") is None:
        print("Installing additional pip dependencies...")
        run("pip install matplotlib ipykernel pandas rasterio rioxarray asf_search opera_utils zarr s3fs dem_stitcher tile_mate contextily h5netcdf h5py ipyleaflet ipywidgets jupyter-leaflet fiona geopandas")
    
    print("Dependencies check complete.")

install_dependencies()


### STABLE PATCH
# NOTE: This patch optimizes NetCDF encoding to ensure exported files are fully 
# compatible with GIS software like QGIS and ArcGIS Pro.
import opera_utils.disp._utils as ut
import opera_utils.disp._download as dl
import opera_utils.disp._reformat as rf
from importlib import reload

# 1. Store a clean copy of the ORIGINAL function 
if not hasattr(ut, '_true_original_func'):
    ut._true_original_func = ut._get_netcdf_encoding

def patched_get_netcdf_encoding(ds, chunks=None):
    encoding = ut._true_original_func(ds, chunks)
    if 'y' in ds.dims and 'x' in ds.dims:
        for var in ds.data_vars:
            if var in encoding and 'chunksizes' in encoding[var]:
                c_y, c_x = encoding[var]['chunksizes'][-2:]
                if ds.y.size < c_y or ds.x.size < c_x:
                    del encoding[var]['chunksizes']
    return encoding

# 2. Apply the patch globally
ut._get_netcdf_encoding = patched_get_netcdf_encoding
dl._get_netcdf_encoding = patched_get_netcdf_encoding
rf._get_netcdf_encoding = patched_get_netcdf_encoding

# 3. Restore metadata-copying logic
reload(dl)
dl._get_netcdf_encoding = patched_get_netcdf_encoding

print("Global optimization patch applied.")

In [ ]:
import base64
import glob
import os
import subprocess
import sys
from getpass import getpass
from io import BytesIO
from netrc import netrc, NetrcParseError
from platform import system

import contextily as cx
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyproj
import rasterio as rasterio
import xarray as xr
from IPython.display import Markdown, display
from ipyleaflet import Map, ImageOverlay, LayersControl, basemaps, TileLayer, basemap_to_tiles, Rectangle, WidgetControl
from ipywidgets import Layout, VBox, Image as IPyImage, HTML, link
from matplotlib import cm, colorbar, ticker
from skimage import exposure
from skimage.color import rgb2gray
### Third party library import -- https://github.com/opera-adt/disp-xr
from disp_xr import product, stack as disp_stack, quality_metrics
import geopandas as gpd

def plot_landslide_overlay(ax, crs_raster, color='black', linewidth=2):
    """
    Optional helper to plot a landslide polygon (KML/GeoJSON) on the provided axes.
    """
    if 'landslide_polygon' in globals() and landslide_polygon and os.path.exists(landslide_polygon):
        try:
            ext = os.path.splitext(landslide_polygon)[1].lower()
            gdf_ls = gpd.read_file(landslide_polygon, driver='KML' if ext=='.kml' else None).to_crs(crs_raster)
            gdf_ls.plot(ax=ax, facecolor='none', edgecolor=color, linewidth=linewidth)
            return True
        except Exception as e:
            return False
    return False

def get_landslide_geojson():
    """Returns a GeoJSON layer for ipyleaflet if it exists."""
    if 'landslide_polygon' in globals() and landslide_polygon and os.path.exists(landslide_polygon):
        try:
            from ipyleaflet import GeoJSON
            import json
            ext = os.path.splitext(landslide_polygon)[1].lower()
            gdf_ls = gpd.read_file(landslide_polygon, driver='KML' if ext=='.kml' else None).to_crs("EPSG:4326")
            return GeoJSON(data=json.loads(gdf_ls.to_json()), style={'color': 'black', 'fillOpacity': 0, 'weight': 2}, name='Landslide Polygon')
        except:
            pass
    return None

## Select Your Area of Interest (AOI)

InSAR data is organized into "frames." A single landslide might be visible from multiple Sentinel-1 orbits (Ascending and Descending). For the best results, you should identify which frame best covers your feature.

1. **Go to ASF Search**: [ASF Search Portal](https://search.asf.alaska.edu/#/?zoom=8.420&center=-122.524,39.621&polygon=POLYGON((-123.5182%2040.0462,-123.4253%2040.0462,-123.4253%2040.0878,-123.5182%2040.0878,-123.5182%2040.0462))&dataset=OPERA-S1&productTypes=DISP-S1&resultsLoaded=true&granule=OPERA_L3_DISP-S1_IW_F09158_VV_20240605T020832Z_20241226T020828Z_v1.0_20250417T234243Z)
2. **Find your Frame**: Draw a box over your landslide, select the **OPERA-S1** dataset, and product type **DISP-S1**.
3. **Copy the WKT**: Select a product and copy the "WKT" (Well-Known Text) geometry. Paste it into the `BBOX` variable below.

> [!NOTE]
> Large features may span across multiple frames. This notebook allows you to easily switch between them by changing the `FRAME_ID`. Each frame's data will be stored in its own dedicated folder (e.g., `subset-ncs_FXXXXX/`).

In [ ]:
# For this example there are actually 5 frames that cover the landslide. Its important to consider all of the frames and pick the best one. This example will go through one at a time.
FRAME_ID = 30713 #DESC
# FRAME_ID = 3327 #DESC
# FRAME_ID = 28758 #ASC
# FRAME_ID = 9158 #ASC
# FRAME_ID = 9159 #ASC


BBOX = "POLYGON((-123.5182 40.0462,-123.4253 40.0462,-123.4253 40.0878,-123.5182 40.0878,-123.5182 40.0462))"
start_date = '2023-04-06'
end_date = '2024-12-31'

# Configure paths for this frame
SUBSET_DIR = f"subset-ncs_F{FRAME_ID}"
EXPORT_DIR = f"export_F{FRAME_ID}"

# Optional: Path to a landslide polygon (KML or GeoJSON)
# If you have a polygon of the landslide area, define it here to see it on all maps.
# landslide_polygon = None
landslide_polygon = "landslide_polygon/BoulderCreek.kml" #Mapped by Mackey and Roering(2011)

## NASA Earthdata Authentication

To download OPERA products, you need a free NASA Earthdata account. The code below will securely handle your credentials.

1. **Login**: Enter your username and password when prompted.
2. **Persistence**: Your credentials will be saved in a hidden `.netrc` file in your home directory, so you won't need to re-enter them in the future.

🔐 Register here: [https://urs.earthdata.nasa.gov/](https://urs.earthdata.nasa.gov/)

In [ ]:
urs = 'urs.earthdata.nasa.gov'
prompts = ['Enter NASA Earthdata Login Username: ',
           'Enter NASA Earthdata Login Password: ']

netrc_name = "_netrc" if system() == "Windows" else ".netrc"
netrc_path = os.path.expanduser(f"~/{netrc_name}")

def write_netrc():
    username = getpass(prompt=prompts[0])
    password = getpass(prompt=prompts[1])
    with open(netrc_path, 'a') as f:
        f.write(f"\nmachine {urs}\n")
        f.write(f"login {username}\n")
        f.write(f"password {password}\n")
    os.chmod(netrc_path, 0o600)

def has_urs_credentials():
    try:
        creds = netrc(netrc_path).authenticators(urs)
        return creds is not None
    except (FileNotFoundError, NetrcParseError):
        return False

if not has_urs_credentials():
    if not os.path.exists(netrc_path):
        open(netrc_path, 'w').close()
    write_netrc()


## Download Cropped DISP-S1 Data

In [ ]:
#Run twice to ensure download success

from opera_utils.disp._download import run_download
from dateutil.parser import parse
from pathlib import Path

# Convert types for the library
start_dt = parse(start_date)
end_dt = parse(end_date)
out_path = Path(SUBSET_DIR)


# Optimize download speed: Use 1 worker for small areas (<0.1°) to avoid overhead, 
# and 4 workers for larger regions to enable parallel downloading.
NUM_DOWNLOAD_WORKERS = 1 if abs(eval(BBOX.split("((")[1].split(",")[0].split()[0]) - eval(BBOX.split("((")[1].split(",")[1].split()[0])) < 0.1 else 4

# This runs the download directly in the notebook's memory, ensuring our patch is used
print("Starting download...")
run_download(
    output_dir=out_path,     # Now passing a Path object
    wkt=BBOX,                # Your WKT string
    frame_id=FRAME_ID,
    start_datetime=start_dt, # Datetime object
    end_datetime=end_dt,     # Datetime object
    num_workers=NUM_DOWNLOAD_WORKERS            # Single worker
)
print("Finished!")

In [ ]:
## Search for failed dowloads and remove. Then go back and repeat download step as needed

import os
import glob
import numpy as np

files = glob.glob(f"{SUBSET_DIR}/*.nc")

if files:
    # Calculate the median file size (the "standard" size)
    sizes = [os.path.getsize(f) for f in files]
    median_size = np.median(sizes)
    print(f"Median file size is: {median_size / 1024:.2f} KB")

    # Delete files that are drastically smaller (e.g., < 50% of the median)
    threshold = median_size * 0.5
    deleted_count = 0

    for f, size in zip(files, sizes):
        if size < threshold:
            print(f"Deleting outlier ({size/1024:.2f} KB): {f}")
            os.remove(f)
            deleted_count += 1

    print(f"Cleanup complete! Removed {deleted_count} outliers.")
else:
    print("No files found to check.")

## Ministack Structure and Layers

This step sets up the workspace for your selected DISP-S1 frame. It then displays a scatter plot of available acquisition pairs, based on the downloaded granules.

In DISP-S1:
- A ministack consists of up to 15 acquisition dates
- All displacements are measured **relative to a fixed reference date**
- The plot shows:
  - **x-axis**: reference date (the same across the stack)
  - **y-axis**: each secondary date paired with that reference
- This results in a vertical alignment of dots in the scatter plot with one column per ministack

Each point represents a pair of dates over which cumulative displacement is measured from the reference.

In [ ]:
# Load displacement info from the directory
nc_dir = SUBSET_DIR
disp_df = product.get_disp_info(nc_dir)

# Group by version
versions = {}
for version in disp_df["version"].unique():
    df = disp_df[disp_df["version"] == version].copy()
    versions[version] = df
    print(f"{version}, size: {df.shape[0]}")

# Plot Date1 vs Date2 for each version
for version, df in versions.items():
    df["date1"] = pd.to_datetime(df["date1"], format="%Y%m%d")
    df["date2"] = pd.to_datetime(df["date2"], format="%Y%m%d")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df["date1"], df["date2"], alpha=0.7, marker="o", color="b")
    ax.set_xlabel("Reference (Start Date)")
    ax.set_ylabel("Secondary (End Date)")
    ax.set_title("Ministacking Structure from the downloaded OPERA DISP-S1 NetCDF files")
    ax.grid(True)
    plt.tight_layout()
    plt.show()
    
print('Files information:')
disp_df

## Visualize the displacement contained in one file

What you'll see on the following maps is the displacement value for the last file in the stack, meaning it shows the cumulative deformation between the two dates.

- The left panel shows the displacement in **UTM coordinates**, which maintains the original projection of the dataset.
- The right panel shows the same displacement data but **reprojected to geographic coordinates (EPSG:4326)**, using latitude and longitude.

- Colors represent motion in the satellite's line-of-sight (LOS) direction  
  - **Negative** indicates motion **away** from the satellite  
  - **Positive** indicates motion **toward** the satellite

In [ ]:
row = disp_df.iloc[-1]
# (Assuming 'row' is already defined from your dataframe)
ds = xr.open_dataset(row.path, chunks={"time": 1})

# 2. Format Dates
d1 = pd.to_datetime(row.date1, format="%Y%m%d").date()
d2 = pd.to_datetime(row.date2, format="%Y%m%d").date()

# 3. Mask and Select Data
da = ds.where(ds.recommended_mask == 1).isel(time=-1).displacement


# vmin, vmax = da.quantile([0.02, 0.98]).values

# limit = abs(da).max().values
# 2. Set vmin and vmax symmetrically
# vmin, vmax = -limit, limit
# print(f"Color scale limits set to: {vmin:.3f} to {vmax:.3f}")
# ---------------------------------------------------------------

# 4. Reproject to Lat/Lon
da.rio.write_crs(pyproj.CRS(ds.spatial_ref.attrs['crs_wkt']).to_epsg(), inplace=True)
da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
da_4326 = da.rio.reproject("EPSG:4326", resampling=rasterio.enums.Resampling.bilinear)

# 5. Plotting
# constrained_layout=True is generally better than tight_layout for 
# handling shared colorbars and keeping subplot sizes consistent.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

# --- Left Plot: UTM ---
# Note: add_colorbar=False is crucial here to keep the plot size consistent
da.plot.imshow(
    ax=ax1, 
    cmap="RdBu_r", 
    # vmin=vmin, 
    # vmax=vmax, 
    add_colorbar=False
)
# Assuming 'plot_landslide_overlay' is your custom function defined elsewhere
plot_landslide_overlay(ax1, da.rio.crs)
ax1.set_title("UTM coordinates")

# --- Right Plot: Lat/Lon ---
# Note: We use the same vmin/vmax so the colors mean the same thing
im = da_4326.plot.imshow(
    ax=ax2, 
    cmap="RdBu_r", 
    # vmin=vmin, 
    # vmax=vmax, 
    add_colorbar=False
)
plot_landslide_overlay(ax2, "EPSG:4326")

# Format Lat/Lon ticks
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.2f}°E"))
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.2f}°N"))
ax2.set_xlabel("Longitude")
ax2.set_ylabel("Latitude")
ax2.set_title("Reprojected to EPSG:4326")

# 6. Shared Colorbar
# We attach the colorbar to the figure, referencing the mappable 'im'
cbar = fig.colorbar(im, ax=[ax1, ax2], label="Line-of-sight displacement [meters]", aspect=30)

# fig.suptitle(title, fontsize=14)
plt.show()


## Stacking the OPERA DISP-S1 Files

The `stack_prod` object is an `xarray.Dataset` that represents the full stack of DISP-S1 data loaded from the downloaded DISP-S1 NetCDF files.

As shown below, the dataset includes:
- **Coordinates**: spatial (x, y) and temporal (time) axes
- **Data variables**: multiple layers beyond just displacement
- **Attributes**: metadata describing the product version, projection, mission, and contact info

While the **`displacement`** variable is the main product of interest (cumulative surface motion in the LOS from the reference date), DISP-S1 products also include a variety of **supporting layers**.

> [!IMPORTANT]
> **Reference Date Context**: 
> - The **first time value** in the `time` coordinate is the **reference date**.
> - **All displacement values are cumulative from this reference date**.
> - Displacement at time index 0 should be ~0 as it represents the reference point in time.
> - Each subsequent time step shows cumulative displacement relative to the first reference date.
> - After you select a **spatial reference point** below, all pixels will be adjusted relative to that location, but the temporal reference remains unchanged.

<font color="red">📌 A more detailed explanation of each layer will be provided later in the notebook.</font>

In [ ]:
# Load the full ministack time series with ALL time steps preserved
stack_prod = disp_stack.combine_disp_product(disp_df)

# IMPORTANT: Do NOT remove the first time step or recompute differences!
# The displacement values are ALREADY cumulative from the reference date (first date in stack_prod["time"]).
# Time index 0 = reference date (zero displacement), Time indices 1+ = cumulative displacements.

reference_date = pd.to_datetime(stack_prod.time.values[0]).strftime("%Y-%m-%d")
epsg = pyproj.CRS(stack_prod.spatial_ref.attrs['crs_wkt']).to_epsg()
print(f'Stack information (displacement is cumulative from reference date {reference_date}):')
print(stack_prod)
print(f'\nTime coordinate (first={reference_date} = reference date with zero displacement):')
print(stack_prod.time.values)

n_ministacks = int(disp_df["date1"].nunique())
print(n_ministacks)


## Selecting a Reference Point for Spatial Referencing

InSAR displacement measurements are **relative in both space and time**. The displacement values in your stack are **cumulative from the temporal reference date**, but to properly visualize and analyze them, you should choose a **local spatial reference point** to remove non-local noise.

The spatial reference point should be:
- Not moving during the observed time period (e.g., solid bedrock, stable infrastructure)  
- Located near your area of interest (to capture differential motion relative to it)
- High Coherence

Once selected, all pixel values in the stack will have this point's displacement subtracted from them, creating a **relative displacement map** where your reference point has ~0 displacement and all other pixels show motion relative to that point.

> [!IMPORTANT]
> **Temporal vs. Spatial Reference**:
> - **Temporal Reference**: The first date in the stack — all values are cumulative from this date
> - **Spatial Reference**: The point you select below — all pixel values will be relative to this location
> - These are independent: the temporal reference stays fixed, only the spatial reference changes

<font color="red">💡 Choosing a good reference is important: a stable or nearby point helps isolate the motion you're investigating. We recommend picking a point using the interactive map below or in another map viewer such as Google Earth. Please ensure the pixel is not masked out.

In [ ]:
from ipyleaflet import FullScreenControl, ImageOverlay, Marker
import base64
from io import BytesIO
from PIL import Image
from ipywidgets import Output, VBox

# Define Tile Layers
mapnik = basemap_to_tiles(basemaps.OpenStreetMap.Mapnik)
mapnik.name = 'OpenStreetMap'

topo = basemap_to_tiles(basemaps.OpenTopoMap)
topo.name = 'OpenTopoMap'

Esritopo = basemap_to_tiles(basemaps.Esri.WorldTopoMap)
Esritopo.name = 'Esri World Topo Map'

Esri = basemap_to_tiles(basemaps.Esri.WorldImagery)
Esri.name = 'Esri World Imagery'

# 1. Create InSAR Displacement Overlay (Transparent NaNs)
# Use 2nd and 98th percentile for robust scaling
vmin, vmax = np.nanpercentile(da_4326.values, [2, 98])

cmap = plt.get_cmap('RdBu_r')
norm = plt.Normalize(vmin, vmax)
data = da_4326.values
rgba_data = cmap(norm(data))

# Set alpha channel to 0 for NaNs (this makes the background transparent)
rgba_data[np.isnan(data), 3] = 0

img = Image.fromarray((rgba_data * 255).astype(np.uint8))
output_img = BytesIO()
img.save(output_img, format='PNG')
url = "data:image/png;base64," + base64.b64encode(output_img.getvalue()).decode()

bounds = [[float(da_4326.y.min()), float(da_4326.x.min())], [float(da_4326.y.max()), float(da_4326.x.max())]]
insar_layer = ImageOverlay(url=url, bounds=bounds, name='InSAR Displacement', opacity=0.8)

# 2. Map initialization
center_lat = da_4326.y.values[len(da_4326.y)//2]
center_lon = da_4326.x.values[len(da_4326.x)//2]

ls_layer = get_landslide_geojson()
m = Map(
    center=(center_lat, center_lon),
    zoom=12,
    scroll_wheel_zoom=True,
    layers = [mapnik, topo, Esritopo, Esri, insar_layer] + ([ls_layer] if ls_layer else []),
    layout=Layout(width='100%', height='600px')
)

m.add_control(LayersControl())
m.add_control(FullScreenControl())

# 3. Click handler with Marker feedback and Output widget
out = Output()
click_marker = Marker(location=(center_lat, center_lon), draggable=False, name="Last Click")
processing = [False]  # Lock to prevent concurrent processing

def handle_interaction(**kwargs):
    if kwargs.get('type') == 'click':
        # If already processing a click, ignore this event
        if processing[0]:
            return
        
        processing[0] = True  # Set lock
        
        try:
            lat, lon = kwargs.get('coordinates')
            click_marker.location = (lat, lon)
            if click_marker not in m.layers:
                m.add_layer(click_marker)
            with out:
                out.clear_output()
                print(f"Selected Coordinates: {lat:.6f}, {lon:.6f}")
        finally:
            # Release lock after a short delay to ignore rapid-fire events
            import time
            time.sleep(0.3)
            processing[0] = False

m.on_interaction(handle_interaction)

display(VBox([m, out]))

### 📍 Interactive Point Selection

Use the map above to select two points for analysis:
1. **Reference Point**: This point defines the "zero" for your displacement measurements. Choose a location that you believe is stable (e.g., solid bedrock away from the landslide).
2. **Target Point (Red)**: This is the location where you want to inspect the displacement time series (e.g., the middle of the landslide).

> [!TIP]
> After clicking, the coordinates will be printed below the map. Copy and paste them into the code cell below.

In [ ]:
# -------------------------------------------------------------------------
# PASTE YOUR COORDINATES HERE
# -------------------------------------------------------------------------
lat_ref, lon_ref = 40.080838, -123.440602  # Blue Marker (Stable Area)
lat_4ts, lon_4ts = 40.062997, -123.486564   # Red Marker (Landslide Analysis)

# -------------------------------------------------------------------------

import pyproj
import xarray as xr

# Convert to UTM for precise spatial selection
epsg = pyproj.CRS(stack_prod.spatial_ref.attrs["crs_wkt"]).to_epsg()
proj = pyproj.Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
ref_x, ref_y = proj.transform(lon_ref, lat_ref)
ts_x, ts_y = proj.transform(lon_4ts, lat_4ts)

# Calibrate the stack: All displacement is now relative to your reference point
# NOTE: Displacement values remain CUMULATIVE from the temporal reference date (first date in stack_prod["time"])
reference_date = pd.to_datetime(stack_prod["time"].values[0]).strftime("%Y-%m-%d")
ref_disp = stack_prod.sel(x=ref_x, y=ref_y, method='nearest').displacement
stack_prod['displacement'] = stack_prod.displacement - ref_disp

print(f"Reference point (stable): ({lat_ref:.4f}, {lon_ref:.4f})")
print(f"Target point (analysis):  ({lat_4ts:.4f}, {lon_4ts:.4f})")
print(f"\nApplied spatial calibration. All displacements are now relative to the reference point, cumulative from {reference_date}.")


## Interpreting Quality Metrics and Results

Now that we've referenced our data, let's look at the movement. However, not all pixels are equally reliable. InSAR measurements can be affected by vegetation, steep terrain, or water.

### How to Read the Map:
- **Color Spectrum (RdBu)**: 
    - <span style="color:red">**Red (Negative Values)**</span>: The ground is moving **away** from the satellite (e.g., downslope movement).
    - <span style="color:blue">**Blue (Positive Values)**</span>: The ground is moving **toward** the satellite.
- **Intensity**: Darker colors represent more significant displacement.

### Quality Layers:
- **Temporal Coherence**: A value from 0 to 1. Areas with coherence **> 0.7** are generally considered highly reliable. Low coherence usually indicates dense vegetation or significant surface changesthat decorrelate the signal.
- **Recommended Mask**: This layer automatically flags pixels that are likely unreliable (water, low coherence). In the visualization below, we apply this mask to focus only on trustworthy data.

In [ ]:
# NOTE: These are cumulative displacement values from the reference date (first date in stack_prod["time"]) through the last date.
# The displacement represents ground motion in the Line-of-Sight (LOS) direction.
def get_baseimage_from_bounds(bounds, source, grayscale=False, gamma=None, log=None, zoom='auto'):
    xmin, ymin, xmax, ymax = bounds
    left, right, bottom, top = cx.plotting._reproj_bb(xmin, xmax, ymin, ymax, 'EPSG:4326', "epsg:3857")
    image, extent = cx.tile.bounds2img(left, bottom, right, top, zoom=zoom, source=source, ll=False)
    image, extent = cx.tile.warp_tiles(image, extent, t_crs='EPSG:4326', resampling=rasterio.enums.Resampling.bilinear)
    if grayscale:
        image = rgb2gray(image[:, :, :3])
    if gamma is not None:
        image = exposure.adjust_gamma(image, gamma)
    if log is not None:
        image = exposure.adjust_log(image, log)
    return image, extent

# Preprocess displacement
buffer_fraction = 0.015
da = stack_prod.isel(time=-1).displacement * 100  # cm
mask = stack_prod.isel(time=-1).recommended_mask
water = stack_prod.isel(time=-1).water_mask
da = da.where((mask == 1) & (water == 1))

# Set CRS and reproject
da.rio.write_crs(epsg, inplace=True)
da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
da_4326 = da.rio.reproject("EPSG:4326", resampling=rasterio.enums.Resampling.bilinear)

# Buffer bounds
xmin, ymin, xmax, ymax = da_4326.rio.bounds()
x_pad = (xmax - xmin) * buffer_fraction
y_pad = (ymax - ymin) * buffer_fraction
buffered_bounds = [xmin - x_pad, ymin - y_pad, xmax + x_pad, ymax + y_pad]
extent = [buffered_bounds[0], buffered_bounds[2], buffered_bounds[1], buffered_bounds[3]]

# Normalize
v = np.nanpercentile(da_4326.values, [2, 98])
vmax = max(abs(v[0]), abs(v[1]))
vmin = -vmax

# Plot setup
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])

# Add basemap
bg, bg_extent = get_baseimage_from_bounds(buffered_bounds, source=cx.providers.Esri.WorldImagery, grayscale=True, gamma=0.9)
ax.imshow(bg, extent=bg_extent, cmap="gray", origin="upper", zorder=1)

# Displacement overlay
ax.imshow(da_4326.values, extent=[xmin, xmax, ymin, ymax], cmap=plt.get_cmap("RdBu_r"), vmin=vmin, vmax=vmax, alpha=0.6, origin="upper", zorder=2)

# Reference point
ax.scatter(lon_ref, lat_ref, color="yellow", s=30, label="Reference Point", zorder=3)
ax.scatter(lon_4ts, lat_4ts, color="magenta", s=30, label="TS Point", zorder=3)

# Colorbar
cbar = plt.colorbar(ax.images[1], ax=ax, shrink=0.6, pad=0.04)
cbar.set_label(f"Displacement [cm] (relative to {pd.to_datetime(stack_prod['time'].values[0]).strftime('%Y-%m-%d')})")

# Add ticks
ax.set_xticks(np.linspace(extent[0], extent[1], 5))
ax.set_yticks(np.linspace(extent[2], extent[3], 5))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.2f}°E"))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.2f}°N"))
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
plot_landslide_overlay(ax, "EPSG:4326")

# Attribution
ax.annotate(
    "Tiles © Esri — Sources: Esri, DigitalGlobe, GeoEye, i-cubed, USDA FSA, USGS, AEX, "
    "Getmapping, Aerogrid, IGN, IGP, swisstopo, and the GIS User Community",
    xy=(0.5, -0.12), xycoords='axes fraction',
    ha='center', va='top', fontsize=6, color='gray'
)


ax.legend()
ax.set_title(f"OPERA DISP-S1 Cumulative Displacement [cm] (relative to {pd.to_datetime(stack_prod['time'].values[0]).strftime('%Y-%m-%d')})")
plt.tight_layout()
plt.show()


## Plot a time series


**Note**:
- If you select the reference point, the plot should display only zeros, because **all displacement is measured relative to this reference point**.
- As you move farther away from the reference point, the data may appear noisier, due to spatial decorrelation and atmospheric effects.
- The **first date** in the time series serves as the **temporal reference**, meaning all displacement values are relative to this initial timestamp.


In [ ]:
# Define the target point

# Get EPSG from dataset
proj = pyproj.Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
x_click, y_click = proj.transform(lon_4ts, lat_4ts)

# Find nearest grid point
x_coords = stack_prod.x.values
y_coords = stack_prod.y.values
x_nearest = x_coords[np.argmin(np.abs(x_coords - x_click))]
y_nearest = y_coords[np.argmin(np.abs(y_coords - y_click))]
distance = np.sqrt((x_click - x_nearest)**2 + (y_click - y_nearest)**2)

# Check distance threshold
if distance > 100:
    print(f"Point too far from valid pixel (distance = {distance:.1f} m)")
else:
    # Check masks
    mask_val = stack_prod.sel(x=x_nearest, y=y_nearest, method="nearest").isel(time=-1).recommended_mask.values
    water_val = stack_prod.sel(x=x_nearest, y=y_nearest, method="nearest").isel(time=-1).water_mask.values

    if mask_val != 1 or water_val != 1:
        print("Selected point is masked out or on water.")
    else:
        # Get displacement
        disp = stack_prod.sel(x=x_nearest, y=y_nearest, method="nearest").displacement
        # Plot time series
        dates = pd.to_datetime(stack_prod.time.values)
        reference_date = dates[0].strftime("%Y-%m-%d")
        plt.figure(figsize=(10, 4))
        plt.scatter(dates, disp.values * 100, s=10, color='black')
        plt.title(f"Displacement Time Series (cm)\nLat={lat_4ts:.4f}, Lon={lon_4ts:.4f}\nRelative to {reference_date}")
        plt.xlabel("Time")
        plt.ylabel(f"LOS Displacement (cm)\n[relative to {reference_date}]")
        plt.grid(True)
        plt.gca().xaxis.set_major_locator(mdates.AutoDateLocator())
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.tight_layout()
        plt.show()


## Inspect DISP-S1 Data Layers and their Attributes

This step prints detailed information about the variables contained in the DISP-S1 data stack.

For each variable (e.g., displacement, coherence, phase similarity), the function will display:
- The **data type**, and **description** of what the layer represents
- Units, if available
- Any relevant metadata or processing notes

In [ ]:
variables = list(stack_prod.data_vars.items())

output_lines = []

for var_name, da in variables[3:]:
    output_lines.append(f"###  Layer: `{var_name}`")
    for attr_name, attr_val in da.attrs.items():
        output_lines.append(f"- **{attr_name}**: {attr_val}")
    output_lines.append("")  # empty line between variables

# Display once, in one cell
display(Markdown("\n".join(output_lines)))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
import pyproj


variables = list(ds.data_vars.keys())
plot_vars = variables[2:] 
num_vars = len(plot_vars)

# Determine grid layout
grid_cols = 4  
grid_rows = int(np.ceil(num_vars / grid_cols))

fig = plt.figure(figsize=(20, 5 * grid_rows))

grid = ImageGrid(fig, 111,          
                 nrows_ncols=(grid_rows, grid_cols),
                 axes_pad=0.7,      
                 share_all=False,
                 cbar_location="right",
                 cbar_mode="each",
                 cbar_size="5%",
                 cbar_pad=0.1,
                 )

epsg = pyproj.CRS(ds.spatial_ref.attrs["crs_wkt"]).to_epsg()

for i, ax in enumerate(grid):
    if i < len(plot_vars):
        var_name = plot_vars[i]
        data_ = ds[var_name]

        # make it 2D and give it spatial metadata
        if "time" in data_.dims:
            data_ = data_.isel(time=-1)
        data_ = data_.rio.write_crs(epsg)
        data_ = data_.rio.set_spatial_dims(x_dim="x", y_dim="y")

        cmap = "viridis"
        vmin = vmax = None
        if var_name in ["displacement", "short_wavelength_displacement"]:
            cmap = "RdBu_r"
            vals = data_.values.ravel()
            vals = vals[~np.isnan(vals)]
            if len(vals) > 0:
                p2, p98 = np.percentile(vals, [2, 98])
                max_val = max(abs(p2), abs(p98))
                vmin, vmax = -max_val, max_val

        im = data_.plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=False)
        ax.cax.colorbar(im).set_label(var_name)
        ax.ticklabel_format(style="sci", axis="both", scilimits=(0, 0))
        ax.set_title(var_name)
        plot_landslide_overlay(ax, data_.rio.crs)
    else:
        ax.set_visible(False)
        ax.cax.set_visible(False)
plt.show()

## Quality Metrics

### Spatial Quality Metrics Overview

The following step summarizes the **spatial quality** of the stack. These percentages help you decide if the data quality is sufficient for your analysis or if you need to adjust your approach (e.g., choosing a different Reference Point or being cautious about noise).

-   **% Persistent Scatterers (PS)**
    -   **Definition:** The fraction of pixels that consistently reflect radar signals (e.g., buildings, rocks).
    -   **Interpretation:**
        -   **High (>10%):** Excellent for long-term monitoring; you have many reliable points.
        -   **Low (<1%):** Common in vegetated or snowy areas. You may need to rely more on the "Recommended Mask" or spatial filtering, as point-by-point analysis might be noisy.

-   **% Valid Time Series Pixels**
    -   **Definition:** The proportion of pixels flagged as "good" by the standard OPERA quality mask (combines coherence, water masking, etc.).
    -   **Interpretation:**
        -   **High (>50%):** You have broad coverage of the landslide.
        -   **Low (<10%):** The area is likely decorrelated (dense vegetation, snow, or rapid motion).
        -   **Action:** Be skeptical of "isolated" pixels. Ensure your Reference Point is within the valid area, otherwise your entire time series may drift.

-   **% Connected Component Valid**
    -   **Definition:** The average percentage of time steps in the ministacks where pixels were part of a valid connected component (i.e., successfully unwrapped relative to their neighbors).
    -   **Interpretation:**
        -   **High (close to 100%):** The phase unwrapping is **temporally stable**. These pixels were reliably connected to the rest of the image throughout the entire time period.
        -   **Low (< 50%):** The phase unwrapping is unstable or frequently fails (breaks) at these locations.
        -   **Action:** If this number is low for your area of interest, the displacement time series might have "phase jumps" or unphysical discontinuities. You should inspect the time series plot carefully for sudden jumps that don't match the expected landslide motion.

In [ ]:
import numpy as np
import pyproj
import matplotlib.pyplot as plt

# Percent of ones helper (drop the ref time step)
def pct_of_ones(arr):
    arr = arr.isel(time=slice(1, None))
    return (arr == 1).sum(dim="time") / arr.sizes["time"] * 100

pct_ps   = pct_of_ones(stack_prod.persistent_scatterer_mask)
pct_mask = pct_of_ones(stack_prod.recommended_mask)
pct_conn = pct_of_ones(stack_prod.connected_component_labels)

# CRS and extent (UTM)
crs_raster = pyproj.CRS(stack_prod.spatial_ref.attrs["crs_wkt"])
extent = [float(stack_prod.x.min()), float(stack_prod.x.max()),
          float(stack_prod.y.min()), float(stack_prod.y.max())]

cmap_pct = "cividis"  # or "Greys", "YlGnBu", "magma"

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
im1 = ax[0].imshow(np.ma.masked_equal(pct_ps,   0), extent=extent, cmap=cmap_pct,
                   clim=[0, 30],  interpolation="nearest", origin="upper")
im2 = ax[1].imshow(np.ma.masked_equal(pct_mask, 0), extent=extent, cmap=cmap_pct,
                   clim=[0, 100], interpolation="nearest", origin="upper")
im3 = ax[2].imshow(np.ma.masked_equal(pct_conn, 0), extent=extent, cmap=cmap_pct,
                   clim=[0, 100], interpolation="nearest", origin="upper")

titles = [
    "Persistent Scatterers (% of stack)",
    "Valid Pixels (% of stack)",
    "Valid Conncomp (% of stack)",
]

for a, im, lbl, title in zip(ax, [im1, im2, im3], ["% PS", "% Valid", "% Valid Conn"], titles):
    fig.colorbar(im, ax=a, orientation="horizontal", label=lbl, pad=0.15)
    a.set_aspect("equal")
    a.set_title(title)
    a.set_xlabel("Easting (m)")
    a.set_ylabel("Northing (m)")
    a.ticklabel_format(style="sci", axis="both", scilimits=(0, 0))
    plot_landslide_overlay(a, crs_raster)

plt.tight_layout()
plt.show()


This following step computes and displays quality indicators for the displacement stack based on metrics derived from the data:

- **Median Temporal Coherence**: Indicates the overall stability of the interferometric signal over time. Higher values suggest more reliable measurements.

- **Median Phase Similarity**: Reflects how consistent the phase is within local neighborhoods. Useful for identifying noisy or unstable areas.

- **Total Number of 2π Phase Jumps**: A diagnostic for phase unwrapping issues. A high count may indicate discontinuities or noise artifacts.

These metrics provide a summary of the stack’s integrity and can help you decide whether to proceed with or filter out unreliable areas.


In [ ]:
import pyproj

# Stats
median_tcoh = quality_metrics.get_stack_stat(stack_prod.temporal_coherence.isel(time=slice(1, None)), mode="median")
median_psim = quality_metrics.get_stack_stat(stack_prod.phase_similarity.isel(time=slice(1, None)), mode="median")
inv_res_sum  = quality_metrics.get_stack_stat(stack_prod.timeseries_inversion_residuals.isel(time=slice(1, None)), mode="sum")
num_2pi_jump = inv_res_sum / (2 * np.pi)

# CRS + extent (UTM)
crs_raster = pyproj.CRS(stack_prod.spatial_ref.attrs["crs_wkt"])
xmin, xmax = float(stack_prod.x.min()), float(stack_prod.x.max())
ymin, ymax = float(stack_prod.y.min()), float(stack_prod.y.max())
extent = [xmin, xmax, ymin, ymax]

fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
im1 = ax[0].imshow(np.ma.masked_equal(median_tcoh, 0), extent=extent, origin="upper",
                   cmap="afmhot", clim=[0, 1], interpolation="nearest")
im2 = ax[1].imshow(np.ma.masked_equal(median_psim, 0), extent=extent, origin="upper",
                   cmap="afmhot", clim=[0, 1], interpolation="nearest")
im3 = ax[2].imshow(np.ma.masked_equal(num_2pi_jump, 0), extent=extent, origin="upper",
                   cmap="plasma", interpolation="nearest")

titles = ["Median temporal coherence", "Median phase similarity", "Number of 2π jumps"]
cbar_labels = ["Temporal coherence (median)", "Phase similarity (median)", "Count of 2π jumps"]

for a, im, t, clbl in zip(ax, [im1, im2, im3], titles, cbar_labels):
    fig.colorbar(im, ax=a, location="bottom", label=clbl, pad=0.12)
    a.set_title(t)
    a.set_xlabel("Easting (m)")
    a.set_ylabel("Northing (m)")
    a.ticklabel_format(style="sci", axis="both", scilimits=(0, 0))
    a.set_aspect("equal")
    plot_landslide_overlay(a, crs_raster)  # landslide polygon in UTM

plt.tight_layout()
plt.show()


The following step computes and plots statistics related to **`shp_counts`**, which represent the number of statistically homogeneous pixels (SHPs) used in the multilooking process at each pixel.

For the selected ministack, this function reports:
- The median number of SHPs per pixel
- The standard deviation of SHP counts across the scene

These metrics help assess the robustness of the distributed scatterer (DS) processing:
- A higher median indicates more reliable local statistics
- High variability (standard deviation) may point to inconsistent signal quality across the scene

In [ ]:
import pyproj
import matplotlib.pyplot as plt

if n_ministacks < 2:
    print("Not enough time steps to compute shp stats.")
else:
    shp_median = quality_metrics.get_stack_stat(stack_prod.shp_counts, mode="median")
    shp_std    = quality_metrics.get_stack_stat(stack_prod.shp_counts, mode="std")

    # CRS + extent (UTM)
    crs_raster = pyproj.CRS(stack_prod.spatial_ref.attrs["crs_wkt"])
    xmin, xmax = float(stack_prod.x.min()), float(stack_prod.x.max())
    ymin, ymax = float(stack_prod.y.min()), float(stack_prod.y.max())
    extent = [xmin, xmax, ymin, ymax]

    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    im1 = ax[0].imshow(np.ma.masked_equal(shp_median, 0), extent=extent, origin="upper",
                       cmap="cividis", interpolation="nearest")
    im2 = ax[1].imshow(np.ma.masked_equal(shp_std, 0), extent=extent, origin="upper",
                       cmap="magma", interpolation="nearest")

    for a, im, title, clbl in zip(
        ax,
        [im1, im2],
        ["SHP Median", "SHP Std Dev"],
        ["SHP (median)", "SHP (std dev)"],
    ):
        fig.colorbar(im, ax=a, orientation="horizontal", pad=0.12, label=clbl)
        a.set_title(title)
        a.set_xlabel("Easting (m)")
        a.set_ylabel("Northing (m)")
        a.ticklabel_format(style="sci", axis="both", scilimits=(0, 0))
        a.set_aspect("equal")
        plot_landslide_overlay(a, crs_raster)

    plt.tight_layout()
    plt.show()


## Velocity Estimation

This cell computes a linear velocity map (in cm/year) from the displacement time series using least squares fitting.
- Fits a linear model: *displacement = velocity × time + intercept*.
- Colors represent motion in the satellite's line-of-sight (LOS) direction  
  - **Negative values** indicates motion **away** from the satellite  
  - **Positive values** indicates motion **toward** the satellite

In [ ]:
# Extract the displacement time series from stack_prod
# NOTE: Velocity is calculated from displacement values that are cumulative from reference date.
# Velocity (cm/year) = cumulative displacement / time span.
disp = stack_prod['displacement'].values  # shape: (nt, ny, nx)
times = stack_prod['time'].values         # datetime64 array
ny, nx = disp.shape[1:]                   # spatial dimensions
nt = disp.shape[0]                        # number of time steps

# Convert time to decimal years
def _decimal_year(dates):
    """Convert datetime64 array to decimal years."""
    import pandas as pd
    dates = pd.to_datetime(dates)
    return dates.year + (dates.dayofyear - 1) / 365.25

tdecimal = _decimal_year(times)  # shape (nt,)
A = np.vstack([tdecimal, np.ones_like(tdecimal)]).T  # shape: (nt, 2)

# Reshape displacement to (nt, ny*nx)
y = disp.reshape(nt, -1)

# Solve least squares for linear fit: displacement = velocity * time + intercept
coef, *_ = np.linalg.lstsq(A, y, rcond=None)  # shape: (2, ny*nx)

# Extract velocity (slope) and reshape back to (ny, nx)
vel = coef[0].reshape(ny, nx).astype(np.float32)

# Create new DataArray in xarray
vel_da = xr.DataArray(vel, dims=("y", "x"), coords={"y": stack_prod.y, "x": stack_prod.x},
    attrs={
        "long_name": "Velocity",
        "units": "m/year",
        "description": "Linear velocity estimated from displacement time series",
        "start_date": str(times[0]),
        "end_date": str(times[-1]),
        "ref_date": str(times[0]),
    },
)

# Add it to a new dataset (or optionally merge with stack_prod)
velocity_ds = xr.Dataset({"velocity": vel_da})


In [ ]:
# Convert velocity to cm/year and apply mask
vel_masked = (vel_da * 100).where((stack_prod.isel(time=-1).recommended_mask == 1) & (stack_prod.isel(time=-1).water_mask == 1))

# Reproject to EPSG:4326
vel_masked.rio.write_crs(epsg, inplace=True)
vel_masked.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
vel_4326 = vel_masked.rio.reproject("EPSG:4326", resampling=rasterio.enums.Resampling.bilinear)

# Buffer bounds
buffer_fraction = 0.015
xmin, ymin, xmax, ymax = vel_4326.rio.bounds()
x_pad = (xmax - xmin) * buffer_fraction
y_pad = (ymax - ymin) * buffer_fraction
buffered_bounds = [xmin - x_pad, ymin - y_pad, xmax + x_pad, ymax + y_pad]
extent = [buffered_bounds[0], buffered_bounds[2], buffered_bounds[1], buffered_bounds[3]]

# Normalize values
v = np.nanpercentile(vel_4326.values, [2, 98])
vmax = max(abs(v[0]), abs(v[1]))
vmin = -vmax

# Use the true reference date for all labels
reference_date = pd.to_datetime(stack_prod['time'].values[0]).strftime('%Y-%m-%d')

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])

# Basemap
bg, bg_extent = get_baseimage_from_bounds(buffered_bounds, source=cx.providers.Esri.WorldImagery, grayscale=True, gamma=0.9)
ax.imshow(bg, extent=bg_extent, cmap="gray", origin="upper", zorder=1)

# Velocity overlay
im = ax.imshow( vel_4326.values, extent=[xmin, xmax, ymin, ymax], cmap="RdBu_r", vmin=vmin, vmax=vmax, alpha=0.6, origin="upper", zorder=2)

# Colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.04)
cbar.set_label(f"Velocity [cm/year] (relative to {reference_date})")

# Ticks and formatting
ax.set_xticks(np.linspace(extent[0], extent[1], 5))
ax.set_yticks(np.linspace(extent[2], extent[3], 5))
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.2f}°E"))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y:.2f}°N"))
ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)

# Attribution
ax.annotate(
    "Tiles © Esri — Sources: Esri, DigitalGlobe, GeoEye, i-cubed, USDA FSA, USGS, AEX, "
    "Getmapping, Aerogrid, IGN, IGP, swisstopo, and the GIS User Community",
    xy=(0.5, -0.12), xycoords='axes fraction',
    ha='center', va='top', fontsize=6, color='gray'
)

ax.set_title(f"OPERA DISP-S1 Estimated Velocity [cm/year] (relative to {reference_date})")
plt.tight_layout()
plt.show()


## Export DISP-S1 Stack for External Use
This final step prepares your displacement time series for export and use in other tools such as **MintPy** and **GIS**.

The process involves two main actions:

1. **Reformat DISP-S1 Files into a Single NetCDF Stack**  
   - Combines multiple DISP-S1 granules (from a ministack) into a single NetCDF file  
   - Uses a defined reference method (`NONE / MEDIAN / BORDER / POINT /  HIGH COHERENCE`) to re-reference all dates consistently  
   - Drops unnecessary variables like `connected_component_labels`, `shp_counts`, etc. to reduce file size  
   - Output: a single file named `disp-output-<frame_id>.nc`.

2. **Convert to MintPy Format**  
   - Uses the exported NetCDF to generate a MintPy-compatible displacement time series  
   - Produces outputs in the `export/` directory  

In [ ]:
# -----------------------------
# DISP-S1 Stack Reformat Script with useful commands
# -----------------------------
# --input-files [PATH [PATH ...]]                                                                                                                                      │
# │       Input DISP-S1 NetCDF files. (required)                                                                                                                         │
# │ --output-name STR                                                                                                                                                    │
# │       Name of the output file.                                                                                                                                       │
# │       Must end in ".nc" or ".zarr". (required)                                                                                                                       │
# │ --out-chunks INT INT INT                                                                                                                                             │
# │       Chunking configuration for output DataArray.                                                                                                                   │
# │       Defaults to (4, 256, 256). (default: 4 256 256)                                                                                                                │
# │ --shard-factors INT INT INT                                                                                                                                          │
# │       For Zarr outputs, sharding configuration for output DataArray.                                                                                                 │
# │       The factors are applied respectively to the chunks sizes in `out_chunks` to                                                                                    │
# │       create fewer output files in the Zarr store.                                                                                                                   │
# │       Defaults to (1, 4, 4). (default: 1 4 4)                                                                                                                        │
# │ --drop-vars {None}|{[STR [STR ...]]}                                                                                                                                 │
# │       list of variable names to drop from the dataset before saving.                                                                                                 │
# │       Example: ["estimated_phase_quality"] (default: None)                                                                                                           │
# │ --apply-solid-earth-corrections, --no-apply-solid-earth-corrections                                                                                                  │
# │       Apply solid earth tide correction to the data.                                                                                                                 │
# │       Default is True. (default: True)                                                                                                                               │
# │ --apply-ionospheric-corrections, --no-apply-ionospheric-corrections                                                                                                  │
# │       Apply ionospheric delay correction to the data.                                                                                                                │
# │       Default is False. (default: False)                                                                                                                             │
# │ --quality-datasets {None}|{[{TEMPORAL_COHERENCE,PHASE_SIMILARITY,PERSISTENT_SCATTERER_MASK,TIMESERIES_INVERSION_RESIDUALS,CONNECTED_COMPONENT_LABELS,                │
# │ RECOMMENDED_MASK,ESTIMATED_PHASE_QUALITY,SHP_COUNTS,WATER_MASK} [...]]}                                                                                              │
# │       Name of the quality datasets to use as a mask when accumulating                                                                                                │
# │       displacement for re-referencing.                                                                                                                               │
# │       If None, no masking is performed.                                                                                                                              │
# │       Must be same length as `quality_thresholds`.                                                                                                                   │
# │       Default is [QualityDataset.RECOMMENDED_MASK], which uses the built-in                                                                                          │
# │       recommended mask. (default: RECOMMENDED_MASK)                                                                                                                  │
# │ --quality-thresholds {None}|{[FLOAT [FLOAT ...]]}                                                                                                                    │
# │       Thresholds for the quality datasets to use as a mask.                                                                                                          │
# │       Must be same length as `quality_datasets`.                                                                                                                     │
# │       Default is [0.5]. (default: 0.5)                                                                                                                               │
# │ --reference-method {NONE,POINT,MEDIAN,BORDER,HIGH_COHERENCE}                                                                                                         │
# │       Reference method to use.                                                                                                                                       │
# │       Default is ReferenceMethod.NONE.                                                                                                                               │
# │       Options are:                                                                                                                                                   │
# │       - ReferenceMethod.NONE: No reference method.                                                                                                                   │
# │       - ReferenceMethod.POINT: Reference point.                                                                                                                      │
# │       - ReferenceMethod.MEDIAN: Full-scene median per date. Excludes water pixels.                                                                                   │
# │       - ReferenceMethod.BORDER: Median of border pixels. Excludes water pixels.                                                                                      │
# │       - ReferenceMethod.HIGH_COHERENCE: Median of high-coherence mask. (default: HIGH_COHERENCE)                                                                     │
# │ --reference-row {None}|INT                                                                                                                                           │
# │       For ReferenceMethod.POINT, row index for point reference. (default: None)                                                                                      │
# │ --reference-col {None}|INT                                                                                                                                           │
# │       For ReferenceMethod.POINT, column index for point reference. (default: None)                                                                                   │
# │ --reference-lon {None}|FLOAT                                                                                                                                         │
# │       For ReferenceMethod.POINT, longitude (in degrees) for point reference. (default: None)                                                                         │
# │ --reference-lat {None}|FLOAT                                                                                                                                         │
# │       For ReferenceMethod.POINT, latitude (in degrees) for point reference. (default: None)                                                                          │
# │ --reference-border-pixels INT                                                                                                                                        │
# │       For ReferenceMethod.BORDER, number of pixels to use for border median.                                                                                         │
# │       Defaults to 3. (default: 3)                                                                                                                                    │
# │ --reference-coherence-threshold FLOAT                                                                                                                                │
# │       For ReferenceMethod.HIGH_COHERENCE, threshold for coherence to use as a mask.                                                                                  │
# │       Defaults to 0.7. (default: 0.7)                                                                                                                                │
# │ --process-chunk-size INT INT                                                                                                                                         │
# │       Chunking configuration for processing DataArray.                                                                                                               │
# │       Defaults to (2048, 2048). (default: 2048 2048)                                                                                                                 │
# │ --do-round, --no-do-round                                                                                                                                            │
# # │       If True, rounds mantissa bits of floating point rasters to compress the data. (default: True)   


In [ ]:
import os
from opera_utils.disp._reformat import reformat_stack
from opera_utils.disp._enums import ReferenceMethod, QualityDataset

# 1. Prepare files
output_dir = EXPORT_DIR
os.makedirs(output_dir, exist_ok=True)
files = sorted(list(Path(SUBSET_DIR).glob("*.nc")))

# 2. Extract Reference point (fallback to high coherence method if not set)
ref_lat = lat_ref if 'lat_ref' in globals() and lat_ref is not None else None
ref_lon = lon_ref if 'lon_ref' in globals() and lon_ref is not None else None

# 3. SELECT METHOD BASED ON AVAILABILITY
if ref_lat and ref_lon:
    method = ReferenceMethod.POINT
    print(f"Using manual Reference Point: {ref_lat}, {ref_lon}")
else:
    method = ReferenceMethod.HIGH_COHERENCE
    print("WARNING: No map point selected. Falling back to HIGH_COHERENCE (automatic).")
    
# 4. Run Reformat
reformat_stack(
    input_files=files,
    output_name=f"{EXPORT_DIR}/disp-output.nc",
    drop_vars=[
        "connected_component_labels", 
        "shp_counts", 
        "timeseries_inversion_residuals"
    ],
    reference_method=method,  # <--- Dynamically selected method
    quality_datasets=[QualityDataset.WATER_MASK],
    reference_border_pixels=3,
    reference_lat=ref_lat,
    reference_lon=ref_lon
)

print(f"\nReformat complete using method: {method.name}")

In [ ]:
# Convert to MintPy
SAMPLE_FILE = sorted(glob.glob(f"{SUBSET_DIR}/*.nc"))[0]
OUTPUT_NAME = "disp-output.nc"

print(f"Converting {OUTPUT_NAME} to MintPy using sample metadata from {SAMPLE_FILE}...")

!python -m opera_utils.disp.mintpy {EXPORT_DIR}/{OUTPUT_NAME} \
    --sample-disp-nc {SAMPLE_FILE} \
    --outdir {EXPORT_DIR}

print("MintPy conversion successful! Files are in the f'{EXPORT_DIR}/' directory.")

In [ ]:
import h5py
import numpy as np
from pathlib import Path
import pyproj

if 'lat_ref' in globals() and 'lon_ref' in globals() and lat_ref is not None and lon_ref is not None:
    epsg = pyproj.CRS(stack_prod.spatial_ref.attrs["crs_wkt"]).to_epsg()
    proj = pyproj.Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
    ref_easting, ref_northing = proj.transform(lon_ref, lat_ref)

    ref_ix = int(np.argmin(np.abs(stack_prod.x.values - ref_easting)))   # column
    ref_iy = int(np.argmin(np.abs(stack_prod.y.values - ref_northing)))  # row

    attrs = {
        "REF_LON": float(ref_easting),  
        "REF_LAT": float(ref_northing),
        "REF_X": ref_ix,
        "REF_Y": ref_iy,
    }

    files = [
        "timeseries.h5",
        "velocity.h5",
        "avgSpatialCoh.h5",
        "timeseries_density.h5",
        "recommended_mask_90thresh.h5",  # adjust name if different
    ]

    for fname in files:
        fpath = Path(EXPORT_DIR) / fname
        if not fpath.exists():
            continue
        with h5py.File(fpath, "r+") as hf:
            for k, v in attrs.items():
                hf.attrs[k] = v
        print(f"Stamped ref metadata into {fpath.name}")
else:
    print("No reference point set; skipping metadata stamp.")


## Build masks for MintPy (currently only water mask is applied)

In [ ]:
import os, xarray as xr, numpy as np, h5py

# --- PARAMETERS ---
output_dir = EXPORT_DIR
TEMP_COH_THRESHOLD = 0.5  # Mask pixels below this coherence ---
VALID_PCT_THRESHOLD = 0.8  # fraction (e.g., 0.8 = 80%) for recommended mask via mini stacks
STRICT_MASK = False        # True = check every date | False = check average

nc_path = os.path.join(output_dir, "disp-output.nc")
ts_path = os.path.join(output_dir, "timeseries.h5")

# 1. Load geometry and open dataset
with h5py.File(ts_path, "r") as f:
    ts_attrs = dict(f.attrs)
ds = xr.open_dataset(nc_path)

def save_mintpy_h5(filename, dataset_name, data, file_type="mask", unit="1"):
    path = os.path.join(output_dir, filename)
    data_val = data.values.squeeze()
    is_mask = file_type in ["mask", "water_mask"]
    dtype = np.int8 if is_mask else np.float32
        
    with h5py.File(path, "w") as f:
        f.attrs.update(ts_attrs)
        f.attrs.update({"FILE_TYPE": file_type, "UNIT": unit})
        f.create_dataset(dataset_name, data=data_val.astype(dtype), compression="gzip")
    print(f"Created 2D Layer: {path} (Type: {file_type})")

# --- STEP 1: Export Standard Science Layers ---
if "average_temporal_coherence" in ds:
    save_mintpy_h5("temporalCoherence.h5", "temporalCoherence", ds["average_temporal_coherence"], file_type="temporalCoherence")
elif "temporal_coherence" in ds:
    save_mintpy_h5("temporalCoherence.h5", "temporalCoherence", ds["temporal_coherence"].mean("time"), file_type="temporalCoherence")

if "phase_similarity" in ds:
    psim = ds["phase_similarity"].mean("time") if "time" in ds["phase_similarity"].dims else ds["phase_similarity"]
    save_mintpy_h5("phaseSimilarity.h5", "phaseSimilarity", psim, file_type="phaseSimilarity")

if "water_mask" in ds:
    save_mintpy_h5("water_mask.h5", "mask", ds["water_mask"])

if "persistent_scatterer_mask" in ds:
    save_mintpy_h5("maskPS.h5", "mask", ds["persistent_scatterer_mask"])

# --- STEP 2: Generate Custom Coherence Mask ---
if STRICT_MASK:
    mask_2d = (ds.temporal_coherence >= TEMP_COH_THRESHOLD).all(dim="time")
    mode_str = "strict"
else:
    avg_coh = ds.average_temporal_coherence if "average_temporal_coherence" in ds else ds.temporal_coherence.mean(dim="time")
    mask_2d = avg_coh >= TEMP_COH_THRESHOLD
    mode_str = "avg"

mask_filename = f"tcoh{str(TEMP_COH_THRESHOLD).replace('.','')}_{mode_str}_mask.h5"
save_mintpy_h5(mask_filename, "mask", mask_2d, file_type="mask")



if "recommended_mask" in ds and "reference_time" in ds:
    # Collapse to one mask per ministack (group by calendar date of reference_time)
    ref_date = pd.to_datetime(ds.reference_time.values).normalize()
    rm = ds.recommended_mask.fillna(0).astype(np.int8)
    rm = rm.assign_coords(reference_date=("time", ref_date))
    per_ms = rm.groupby("reference_date").max("time")  # (n_ms, y, x)

    # Percent of ministacks where pixel is recommended
    ms_pct = per_ms.mean("reference_date") * 100
    save_mintpy_h5("recommended_mask_ministack_pct.h5", "recommendedMaskPct",
                   ms_pct, file_type="percent", unit="%")

    # Thresholded mask
    pct_thresh = int(VALID_PCT_THRESHOLD * 100)
    ms_mask_thresh = ms_pct >= pct_thresh
    save_mintpy_h5(f"recommended_mask_{pct_thresh}thresh.h5", "recommendedMask",
                   ms_mask_thresh, file_type="mask", unit="1")

    print(f"Ministacks (unique reference dates): {per_ms.sizes['reference_date']}")
else:
    print("No recommended_mask/reference_time found; skipping ministack-based mask.")


print(f"\nSuccess! All supplementary layers and your custom {mode_str} mask are ready in /{output_dir}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pyproj
from mpl_toolkits.axes_grid1 import ImageGrid
import numpy as np

# Build a date-only coord (drops time-of-day)
ref_date = pd.to_datetime(stack_prod.reference_time.values).normalize()
rm = stack_prod.recommended_mask.fillna(0).astype(np.int8)
rm = rm.assign_coords(reference_date=("time", ref_date))

# One mask per reference date
per_ms = rm.groupby("reference_date").max("time")  # dims: reference_date, y, x

# Spatial extent/CRS
crs_raster = pyproj.CRS(stack_prod.spatial_ref.attrs["crs_wkt"])
extent = [float(stack_prod.x.min()), float(stack_prod.x.max()),
          float(stack_prod.y.min()), float(stack_prod.y.max())]

n_ms = per_ms.sizes["reference_date"]
fig = plt.figure(figsize=(6 * n_ms, 6))
grid = ImageGrid(fig, 111, nrows_ncols=(1, n_ms), axes_pad=0.5,
                 share_all=False, cbar_location="right", cbar_mode="each",
                 cbar_size="5%", cbar_pad=0.1)

for i, ax in enumerate(grid):
    da = per_ms.isel(reference_date=i)
    im = ax.imshow(da, extent=extent, origin="upper", cmap="Greys",
                   vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Recommended Mask\n{str(per_ms.reference_date.values[i])[:10]}")
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="sci", axis="both", scilimits=(0, 0))
    ax.set_aspect("equal")
    ax.cax.colorbar(im).set_label("Mask (0/1)")
    plot_landslide_overlay(ax, crs_raster)

plt.tight_layout()
plt.show()


## 📂 GIS Export & Data Visualization

Your analysis is complete! Now, let's export the data for use in professional GIS tools or for presentations.

### 🌍 1. GeoTIFF for QGIS/ArcGIS
The code below generates two GeoTIFFs:
- **Basic Mask**: Only removes water.
- **Quality Mask (Recommended)**: Removes unreliable pixels based on quality metrics.

> [!TIP]
> When loading into GIS, set the **Symbology** to specialized colormaps like `RdBu` (Red-White-Blue) to clearly see motion towards/away from the satellite.

### 🎬 2. Time Series Animation (GIF)
To see how the landslide evolved over months, we stitch all acquisition dates into a single animation.

In [ ]:
# Ensure EXPORT_DIR exists
os.makedirs(EXPORT_DIR, exist_ok=True)

# Define output filenames using FRAME_ID for uniqueness
velocity_tif_basic = os.path.join(EXPORT_DIR, f"velocity_F{FRAME_ID}_waterMsk.tif")
velocity_tif_cleaned = os.path.join(EXPORT_DIR, f"velocity_F{FRAME_ID}_recommendedMsk.tif")

print(f"Exporting GeoTIFFs to: {EXPORT_DIR}")

# 1. Basic Mask (Water only)
# Scale velocity to cm/year (multiply by 100)
vel_basic = (vel_da * 100).where(stack_prod.isel(time=-1).water_mask == 1)
vel_basic.rio.write_crs(epsg, inplace=True)
vel_basic.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
vel_basic.rio.to_raster(velocity_tif_basic)
print(f"Saved Basic Mask GeoTIFF: {velocity_tif_basic}")

# 2. Quality Mask (Recommended Mask)
# Scale velocity to cm/year
vel_cleaned = (vel_da * 100).where(
    (stack_prod.isel(time=-1).recommended_mask == 1) & 
    (stack_prod.isel(time=-1).water_mask == 1)
)
vel_cleaned.rio.write_crs(epsg, inplace=True)
vel_cleaned.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
vel_cleaned.rio.to_raster(velocity_tif_cleaned)
print(f"Saved Cleaned Mask GeoTIFF: {velocity_tif_cleaned}")

In [ ]:
# Export PNGs and Create GIF Animation
import os
import glob
import h5py
from PIL import Image  # Standard library for GIF creation

# --- CONFIG ---
output_dir = EXPORT_DIR
pictures_dir = os.path.join(output_dir, "pics")
os.makedirs(pictures_dir, exist_ok=True)

# Ensure ds_full is loaded
if 'ds_full' not in locals():
    input_nc = os.path.join(output_dir, "disp-output.nc")
    import xarray as xr
    ds_full = xr.open_dataset(input_nc)

# 1. Load Data & Apply Mask (only if not already defined)
if 'mask_da' not in locals():
    mask_path = os.path.join(output_dir, "tcoh05_avg_mask.h5")
    with h5py.File(mask_path, "r") as f:
        mask = f["mask"][...]
    mask_da = xr.DataArray(mask, dims=("y", "x"), coords={"y": ds_full.y, "x": ds_full.x})
if 'disp_masked_raw' not in locals():
    disp_masked_raw = ds_full["displacement"].where(mask_da == 1)

# 2. Timing & Color Limits
# Use the true reference date (first date in ds_full["time"]) for all labels and outputs
dates_f = ds_full["time"].dt.strftime("%Y%m%d").values
dates_t = ds_full["time"].dt.strftime("%Y-%m-%d").values
v = np.nanpercentile(disp_masked_raw[-1], [1, 99])
max_abs = max(abs(v[0]), abs(v[1]))
vmin, vmax = -max_abs, max_abs
extent = [float(ds_full.x.min()), float(ds_full.x.max()), float(ds_full.y.min()), float(ds_full.y.max())]

# 3. Polygon Loader (only if not already loaded)
plot_polygon = False
if 'gdf_ls' in locals() and gdf_ls is not None:
    plot_polygon = True
elif 'landslide_polygon' in locals() and landslide_polygon and os.path.exists(landslide_polygon):
    try:
        import geopandas as gpd
        import fiona
        fiona.drvsupport.supported_drivers['KML'] = 'rw'
        from pyproj import CRS
        crs_raster = CRS.from_wkt(ds_full.spatial_ref.crs_wkt)
        ext = os.path.splitext(landslide_polygon)[1].lower()
        gdf_ls = gpd.read_file(landslide_polygon, driver='KML' if ext=='.kml' else None).to_crs(crs_raster)
        plot_polygon = True
        print(f"Loaded polygon: {landslide_polygon}")
    except Exception as e:
        print(f"Polygon skipped: {e}")

# 4. Generate PNGs
png_files = []
print("Generating PNG frames...")
png_files = []
for i in range(len(dates_f)):
    fig, ax = plt.subplots(figsize=(8, 6), dpi=100, constrained_layout=True)
    im = ax.imshow(disp_masked_raw[i].values, cmap="RdBu", origin="upper",
                   vmin=vmin, vmax=vmax, extent=extent)
    if plot_polygon:
        gdf_ls.boundary.plot(ax=ax, edgecolor="black", linewidth=1.5, zorder=5)

    ax.set_title(f"Cumulative Displacement (m) from {reference_date} (ref) to {dates_t[i]}",
                 pad=10)
    cbar = fig.colorbar(im, ax=ax, location="right", fraction=0.05, pad=0.04)
    cbar.set_label(f"LOS Displacement (m)\n[relative to {reference_date}]")

    p = os.path.join(pictures_dir, f"frame_{i:03d}.png")
    fig.savefig(p, dpi=100)  # no bbox_inches="tight" → fixed canvas
    plt.close(fig)
    png_files.append(p)


# 5. Stitch into GIF
gif_path = os.path.join(output_dir, f"OPERA-DISP-S1_Frame_{FRAME_ID}_TimeSeries.gif")
print(f"Stitching {len(png_files)} frames into GIF...")
images = [Image.open(f) for f in png_files]
images[0].save(gif_path, save_all=True, append_images=images[1:], duration=250, loop=0)
print(f"Success! View your animation at: {gif_path}")


### ✅ Conclusion and Next Steps

Congratulations! You've successfully processed OPERA DISP-S1 data to track ground displacement. 

**Key Takeaways:**
- **Referencing is Key**: Selecting a stable reference point is essential for accurate results.
- **Quality Matters**: Always check the `quality metrics` before relying on high-magnitude movement signals.
- **Multi-Frame Ready**: If your landslide is covered by more than one frame, you can now change the `FRAME_ID` at the top and re-run to append more data.

**Explore Further**:
- Check out the [OPERA Project Website](https://www.jpl.nasa.gov/go/opera) for more datasets.

---
*End of workflow.*